# Notebook Information A2 Part 1: Feature Importance
1. Research methods of feature imporance in scikit-learn
2. Using a model from A1 & a method of feature importance. Find 50 most important features
2. Description of feature importance method used. Comment on output, 50 most important features

# Methods of Feature Importance
___
### Tree-Based Models:
- Forest of Trees means that tree-based models have built-in methods of calculating feature importance.
- Built-in method: model.feature_importances_
- Models inclue: Decision Tree, Random Forests and Gradient Boosted Trees

### Permutation Importance:
- Randomly shuffle features in model: rearrange chosen feature keeping others the same 
- E.g. Predict BMI: Swap around height values keeping weight, gender the same.
- Test model performance after each "shuffling". Whichever shuffled feature leads to the biggest decrease, that is the most important
- Works for all models

### Coefficients Importance:
- Works for linear models
- How linear models work: y = a1x1 + a2x2 + a3x3 + ... + aNxN (y = prediction | a = coefficients | x = features)
- The coefficient (a) with the largets absolute value (positive or negative) is represents the most important feature
- E.g. (Height a1x1 where a1 = 1.2 | Weight = a2x2 where a2 = -1.6) Weight is the most important feature
- Models include: Linear regression, Logistic Regression

# Feature Importance on Multinomial Naive Bayes Model
___

In [1]:
pip install scikit-learn

Note: you may need to restart the kernel to use updated packages.


In [2]:
pip install datasets

Note: you may need to restart the kernel to use updated packages.


## Create Multinomial Naive Bayes Model

In [4]:
from datasets import load_dataset
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB

imdb_dataset = load_dataset("imdb")["train"]

train_data = []
train_data_labels = []
for item in imdb_dataset:
    train_data.append(item["text"])
    train_data_labels.append(item["label"])

vectorizer = TfidfVectorizer(analyzer="word", max_features=5000, lowercase=True)
features = vectorizer.fit_transform(train_data).toarray()

x_train, x_val, y_train, y_val = train_test_split(features, train_data_labels, train_size=0.9, random_state=123)
model = MultinomialNB()
model = model.fit(X=x_train, y=y_train)

from sklearn.metrics import accuracy_score

y_pred = model.predict(x_val)
print(accuracy_score(y_val, y_pred))

0.8396


## Feature Importance Method: Permutation Importance

In [6]:
import numpy as np

baseline_accuracy = accuracy_score(y_val, y_pred)

# Function to calculate permutation importance
def permutation_importance(model, X_val, y_val, baseline_accuracy):
    feature_importances = []
    X_val_copy = X_val.copy()

    for col in range(X_val.shape[1]):
        # Shuffle one feature column
        np.random.shuffle(X_val_copy[:, col])

        # Predict with shuffled data
        y_pred_permuted = model.predict(X_val_copy)

        # Measure performance with shuffled data
        permuted_accuracy = accuracy_score(y_val, y_pred_permuted)

        # Calculate importance as the drop in accuracy
        importance = baseline_accuracy - permuted_accuracy
        feature_importances.append(importance)

        # Restore the original feature values for the next iteration
        X_val_copy[:, col] = X_val[:, col]

    return feature_importances

# Calculate permutation importance for the model
importances = permutation_importance(model, x_val, y_val, baseline_accuracy)

# Rank features by importance
sorted_importances = sorted(enumerate(importances), key=lambda x: x[1], reverse=True)

In [43]:
import pandas as pd

# Extract the top 50 most important features
top_50_features = sorted_importances[:50]

# Map feature indices to their corresponding feature names
feature_names = vectorizer.get_feature_names_out()

top_50_feature_names = [feature_names[i[0]] for i in top_50_features]
top_50_feature_importances = [i[1] for i in top_50_features]

most_important_50_df = pd.DataFrame({'Word': top_50_feature_names, 'Importance': top_50_feature_importances})
most_important_50_df

,Word,Importance
0,worst,0.0028
1,attempt,0.0024
2,appreciate,0.0020
3,favorite,0.0020
4,waste,0.0020
5,boring,0.0016
6,days,0.0016
7,horrible,0.0016
8,liked,0.0016
9,oh,0.0016


# Explanation of Feature Importance on Multinomial Naive Bayes Model
I tested feature importance on the Multinomial Naive Bayes model I used in A1. As Multinomial Naive Bayes is neither a tree or linear model we must use permutation to get feature importance. This means shuffling our moving around each feature and seeing which feature causes the biggest decrease in model performance. Some words' importance is 0, some have insignificant importances and some show us that they do indeed contribute to our model's predictions. Upon completing the permutation process we are left with a sorted array of feature importances. This array gives us with the position of the word along with its feature importance. i.e (48, 0.5) would mean word at position 48 has feature importance 0.5. We can simply get the word using indexing with our vectorizer. We then make a list containing just the importances and create a dataframe of the words and their importances for viewing and analysis.

**Key Points from Top 50 Feature Importances**
- The word "worst has the greatest feature importance in our model. Its importance is 0.0028. I would assume this word to have negative influences which would not surprise as it is a highly negative word, which unless sarcastically is difficult to use in a positive review.
- The word with the 2nd greatest feature importance in out model is "attempt", which suprises me. Its importance is 0.0028. I cannot say for certain wether the word "attempt" has negative or positive influences as I can see it being used in both contexts. e.g. positve: "the director made a great attempt at capturing emotion in this film"